# 01 — Preprocessing CT Scan (LIDC-IDRI → DenseNet121)

Notebook **pipeline utama**. Mengubah satu axial slice dari crop 3D menjadi
tensor input DenseNet121.

**Isi notebook ini:** konfigurasi, pembacaan annotation, pembacaan DICOM,
konversi HU, lung window, crop, resize, 3 kanal, dan preprocessing DenseNet121,
disertai validasi pada beberapa sample.

**Di luar notebook ini (sengaja):**

| Topik | Lokasi |
|---|---|
| Audit konvensi bbox, kandidat margin, statistik window | `scripts/04_validate_preprocessing.py` |
| Unduh DICOM dari IDC | `scripts/02_download_dataset.py` |
| Ekstraksi annotation | `scripts/03_extract_pylidc_nodules.py` |
| Training, split, augmentation, Grad-CAM | belum dikerjakan |

### Yang sudah dikunci (tidak diubah di notebook ini)

| Parameter | Nilai |
|---|---|
| Window | lung, `HU_MIN = -1350`, `HU_MAX = 150` |
| BBox | `BBox_X` = baris (row), `BBox_Y` = kolom (column) |
| Slice | `middle_index = (n_slices - 1) // 2` |
| Framework | TensorFlow / Keras |
| Model | `DenseNet121(weights="imagenet")` |
| Input | `(224, 224, 3)` channels_last (HWC) |
| Preprocessing | `tf.keras.applications.densenet.preprocess_input` |
| Margin konteks | **BELUM DIKUNCI** — `0.0 mm` hanya placeholder |

## 1. Configuration

Semua path diturunkan dari `PROJECT_ROOT` yang dicari dari lokasi notebook,
sehingga notebook portabel antara laptop, PC Lab, dan `git clone` baru.

In [13]:
# --- CELL 1.1 - IMPORTS -------------------------------------------------
import sys
import platform
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from skimage.transform import resize as sk_resize

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from IPython.display import display
except ImportError:                      # di luar Jupyter
    def display(obj):
        print(obj)

%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


In [14]:
# --- CELL 1.2 - PARAMETER TERKUNCI ---------------------------------------
# Nilai-nilai ini adalah keputusan metodologi. UBAH HANYA lewat
# persetujuan eksperimen terpisah, bukan di notebook ini.

# Window (TAHAP 2) - global, tidak adaptif
WINDOW_HU_MIN = -1350.0
WINDOW_HU_MAX = 150.0

# Konvensi bbox - divalidasi terhadap data LIDC (lihat catatan di bawah)
BBOX_XY_CONVENTION = "transposed"        # BBox_X = baris, BBox_Y = kolom

# Aturan slice (TAHAP 3) - geometris
SLICE_SELECTION = "geometric_lower_middle"

# Input model
TARGET_SIZE = (224, 224)
NUM_CHANNELS = 3
CHANNEL_LAYOUT = "channels_last"         # DenseNet121 Keras: (H, W, C)
INPUT_SHAPE_HWC = (TARGET_SIZE[0], TARGET_SIZE[1], NUM_CHANNELS)
INPUT_SHAPE_NHWC = (None, TARGET_SIZE[0], TARGET_SIZE[1], NUM_CHANNELS)

# PLACEHOLDER - BUKAN KEPUTUSAN METODOLOGI.
# Margin konteks BELUM dikunci. Nilai 0 mm hanya agar pipeline sample
# dapat dijalankan. Kandidat margin diaudit terpisah di
# scripts/04_validate_preprocessing.py
CONTEXT_MARGIN_MM = 0.0
MARGIN_STATUS = "BELUM DIKUNCI"

# Jumlah sample yang divisualikan
N_SAMPLE_FIGURES = 2

# ---------------------------------------------------------------- paths
ANNOTATION_RELPATH = Path("dataset/02_annotations/pylidc_nodules_extracted.xlsx")
RAW_RELPATH = Path("dataset/01_raw")
FIGURE_RELPATH = Path("results/preprocessing_sample")
SAMPLE_INDEX_RELPATH = Path("dataset/04_preprocessed/sample")

# Seri yang dipakai untuk sample dan validasi
SAMPLE_SERIES_UID = "1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016904899083676284739"

# Toleransi verifikasi preprocessing terhadap fungsi Keras
KERAS_TOLERANCE = 1e-6

# --------------------------------------------------- PROJECT_ROOT
def find_project_root(marker: Path, start: Path | None = None) -> Path:
    """Cari folder repository yang memuat `marker`, tanpa path hard-code."""
    base = Path(start).resolve() if start else Path.cwd().resolve()
    for candidate in [base, *base.parents]:
        if (candidate / marker).exists():
            return candidate
    searched = "\n".join(f"  - {p}" for p in [base, *base.parents])
    raise FileNotFoundError(
        f"PROJECT_ROOT tidak ditemukan.\n"
        f"Notebook mencari folder yang memuat: {marker}\n\n"
        f"Direktori yang sudah diperiksa:\n{searched}\n\n"
        f"Pastikan notebook dijalankan dari dalam repository."
    )


PROJECT_ROOT = find_project_root(ANNOTATION_RELPATH)
ANNOTATION_FILE = PROJECT_ROOT / ANNOTATION_RELPATH
RAW_DIR = PROJECT_ROOT / RAW_RELPATH
FIGURE_DIR = PROJECT_ROOT / FIGURE_RELPATH
SAMPLE_INDEX_DIR = PROJECT_ROOT / SAMPLE_INDEX_RELPATH

print("KONFIGURASI")
print("-" * 70)
print(f"Python              : {sys.version.split()[0]}  ({platform.system()} {platform.release()})")
print(f"PROJECT_ROOT        : {PROJECT_ROOT}")
print(f"Window (lung)       : HU_MIN={WINDOW_HU_MIN}  HU_MAX={WINDOW_HU_MAX}")
print(f"  setara level/width: {(WINDOW_HU_MIN + WINDOW_HU_MAX) / 2:.0f} / {WINDOW_HU_MAX - WINDOW_HU_MIN:.0f}")
print(f"Konvensi bbox       : {BBOX_XY_CONVENTION}  (BBox_X=baris, BBox_Y=kolom)")
print(f"Aturan slice        : {SLICE_SELECTION}  -> (n-1)//2")
print(f"Ukuran input        : {INPUT_SHAPE_HWC}  ({CHANNEL_LAYOUT})")
print(f"Batch input model   : {INPUT_SHAPE_NHWC}")
print(f"Margin konteks      : {CONTEXT_MARGIN_MM:g} mm  ->  {MARGIN_STATUS}")


KONFIGURASI
----------------------------------------------------------------------
Python              : 3.12.10  (Windows 11)
PROJECT_ROOT        : C:\230016_rivan\source-code2020
Window (lung)       : HU_MIN=-1350.0  HU_MAX=150.0
  setara level/width: -600 / 1500
Konvensi bbox       : transposed  (BBox_X=baris, BBox_Y=kolom)
Aturan slice        : geometric_lower_middle  -> (n-1)//2
Ukuran input        : (224, 224, 3)  (channels_last)
Batch input model   : (None, 224, 224, 3)
Margin konteks      : 0 mm  ->  BELUM DIKUNCI


## 2. Load Annotation Metadata

File annotation adalah sumber kebenaran koordinat. **File ini tidak diubah,
tidak ditulis ulang, dan tidak diimpor ulang dari `pylidc`.**

In [15]:
# --- CELL 2 - BACA ANNOTATION --------------------------------------------
REQUIRED_COLUMNS = [
    "PatientID", "NoduleID", "ClusterIndex", "SeriesInstanceUID",
    "BBox_X_Min", "BBox_X_Max", "BBox_Y_Min", "BBox_Y_Max",
    "BBox_Z_Min", "BBox_Z_Max",
]

if not ANNOTATION_FILE.is_file():
    raise FileNotFoundError(f"File annotation tidak ditemukan:\n  {ANNOTATION_FILE}")

ann = pd.read_excel(ANNOTATION_FILE)

missing = [c for c in REQUIRED_COLUMNS if c not in ann.columns]
if missing:
    raise KeyError(
        f"Kolom wajib tidak ada di annotation: {missing}\n"
        f"Kolom yang tersedia: {list(ann.columns)}"
    )

ann["SeriesInstanceUID"] = ann["SeriesInstanceUID"].astype(str).str.strip()

print(f"File    : {ANNOTATION_RELPATH}")
print(f"Baris   : {len(ann)}")
print(f"Kolom   : {len(ann.columns)}")
print(f"Seri    : {ann['SeriesInstanceUID'].nunique()}")
print(f"\nDistribusi label (informasi, TIDAK dipakai sebagai filter):")
print(ann["Label"].value_counts().to_string())


File    : dataset\02_annotations\pylidc_nodules_extracted.xlsx
Baris   : 837
Kolom   : 18
Seri    : 269

Distribusi label (informasi, TIDAK dipakai sebagai filter):
Label
Benign       381
Exclude      313
Malignant    143


## 3. Discover DICOM Series

Menelusuri `dataset/01_raw/` dan memetakan `SeriesInstanceUID` → file DICOM,
lalu memilih seri sample.

In [16]:
# --- CELL 3 - INVENTORY DICOM LOKAL --------------------------------------
import os


def discover_series(raw_dir: Path) -> dict[str, list[Path]]:
    """Petakan SeriesInstanceUID -> daftar file DICOM di bawah raw_dir."""
    found: dict[str, list[Path]] = {}
    if not raw_dir.is_dir():
        return found
    for dirpath, dirnames, filenames in os.walk(raw_dir):
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        files = [Path(dirpath) / f for f in filenames
                 if Path(f).suffix.lower() in {".dcm", ".dicom", ""}]
        if not files:
            continue
        try:
            head = pydicom.dcmread(sorted(files)[0], stop_before_pixels=True,
                                   force=True)
            uid = str(getattr(head, "SeriesInstanceUID", "") or "").strip()
        except Exception:                                  # noqa: BLE001
            uid = ""
        found.setdefault(uid or Path(dirpath).name, []).extend(files)
    return {uid: sorted(v) for uid, v in found.items()}


local_series = discover_series(RAW_DIR)
ann_uids = set(ann["SeriesInstanceUID"])
matched_uids = [uid for uid in local_series if uid in ann_uids]

print(f"Folder DICOM        : {RAW_RELPATH}")
print(f"Seri ditemukan      : {len(local_series)}")
print(f"Seri bertannotation : {len(matched_uids)}")

if not matched_uids:
    raise FileNotFoundError(
        f"Tidak ada seri DICOM yang cocok dengan annotation.\n"
        f"Folder yang diperiksa : {RAW_DIR}\n"
        f"Unduh DICOM           : scripts/02_download_dataset.py"
    )

# --- pilih seri sample ----------------------------------------------------
sample_uid = SAMPLE_SERIES_UID.strip()
if sample_uid not in local_series:
    fallback = matched_uids[0]
    print(f"\nPerhatian: SAMPLE_SERIES_UID tidak ada di disk.")
    print(f"           memakai seri pertama yang tersedia: {fallback}")
    sample_uid = fallback

sample_ann = (ann[ann["SeriesInstanceUID"] == sample_uid]
              .sort_values("ClusterIndex").reset_index(drop=True))
if sample_ann.empty:
    raise ValueError(f"SeriesInstanceUID tidak ada di annotation: {sample_uid}")

dicom_files = local_series[sample_uid]
print(f"\nSeri sample         : {sample_uid}")
print(f"PatientID           : {sample_ann['PatientID'].iloc[0]}")
print(f"Jumlah cluster      : {len(sample_ann)}")
print(f"Jumlah file DICOM   : {len(dicom_files)}")


Folder DICOM        : dataset\01_raw
Seri ditemukan      : 306
Seri bertannotation : 269

Seri sample         : 1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016904899083676284739
PatientID           : LIDC-IDRI-0188
Jumlah cluster      : 10
Jumlah file DICOM   : 113


## 4. CT Series Loader

Kelas `CtSeries` membaca header sekali dan memuat pixel on-demand.

**Catatan urutan slice (`k`).** `k` adalah urutan **ascending proyeksi
`ImagePositionPatient` terhadap normal slice**. Jadi `k = 0` berarti *proyeksi
terkecil*, **bukan** "slice inferior". Arah normal slice tidak diverifikasi
di notebook ini, sehingga istilah *superior* / *inferior* tidak dipakai sebagai
klaim umum.

In [17]:
# --- CELL 4 - KELAS PEMBACA CT SERIES -------------------------------------
class CtSeries:
    """Pembaca minimal satu CT series: header sekali, pixel on-demand."""

    def __init__(self, uid: str, files: list[Path]):
        self.uid = str(uid)
        self.files = sorted(files)
        self.headers = [pydicom.dcmread(f, stop_before_pixels=True, force=True)
                        for f in self.files]

        h0 = self.headers[0]
        if str(h0.Modality).strip().upper() != "CT":
            raise ValueError(f"Modality {h0.Modality!r} bukan CT (series {uid}).")

        self.n_rows, self.n_cols = int(h0.Rows), int(h0.Columns)
        # DICOM PixelSpacing = [row_spacing, col_spacing].
        # Dua nilai ini TIDAK diasumsikan sama dan tidak dirata-ratakan.
        self.row_spacing, self.col_spacing = (float(v) for v in h0.PixelSpacing)
        if self.row_spacing <= 0 or self.col_spacing <= 0:
            raise ValueError(f"PixelSpacing tidak valid pada series {uid}.")
        self.slice_thickness = float(getattr(h0, "SliceThickness", 0.0) or 0.0)
        self.rescale_slope = float(getattr(h0, "RescaleSlope", 1.0))
        self.rescale_intercept = float(getattr(h0, "RescaleIntercept", 0.0))

        for key in ("ImageOrientationPatient", "PixelSpacing", "SliceThickness"):
            values = {
                (tuple(np.asarray(getattr(h, key), float).ravel())
                 if key != "SliceThickness" else (float(getattr(h, key)),))
                for h in self.headers
            }
            if len(values) != 1:
                raise ValueError(f"{key} tidak seragam pada series {uid}.")

        iop = np.asarray(h0.ImageOrientationPatient, dtype=float)
        normal = np.cross(iop[0:3], iop[3:6])
        normal = normal / np.linalg.norm(normal)
        positions = np.asarray([h.ImagePositionPatient for h in self.headers],
                               dtype=float)
        # k = urutan ascending proyeksi onto normal slice.
        # k=0 = proyeksi terkecil (bukan "inferior" secara anatomis).
        self.slice_normal = normal
        self.projections = positions @ normal
        self.order = np.argsort(self.projections, kind="stable")

        diffs = np.diff(self.projections[self.order])
        if len(diffs) and diffs.min() < -1e-6:
            raise ValueError(f"Urutan slice tidak monoton pada series {uid}.")
        self.slice_spacing_mm = (float(diffs.mean()) if len(diffs)
                                 else self.slice_thickness)

    @property
    def n_slices(self) -> int:
        return len(self.order)

    @property
    def spacing_is_isotropic(self) -> bool:
        """True bila row_spacing == col_spacing (LIDC-CT umumnya)."""
        return abs(self.row_spacing - self.col_spacing) < 1e-9

    def file_for_k(self, k: int) -> Path:
        return self.files[int(self.order[k])]

    def read_pixels(self, k: int) -> np.ndarray:
        """Baca pixel_array mentah slice ke-k (belum rescale)."""
        if not 0 <= k < self.n_slices:
            raise IndexError(f"slice k={k} di luar 0..{self.n_slices - 1}")
        return pydicom.dcmread(self.file_for_k(k), force=True).pixel_array


## 5. HU Conversion (TAHAP 1)

```
HU = pixel_array * RescaleSlope + RescaleIntercept
```

In [18]:
# --- CELL 5 - TAHAP 1: HU ------------------------------------------------
def to_hounsfield(pixel_array: np.ndarray, rescale_slope: float,
                  rescale_intercept: float) -> np.ndarray:
    """TAHAP 1 - pixel_array DICOM -> Hounsfield Unit (float32)."""
    arr = pixel_array
    if arr.dtype.itemsize > 2:
        arr = arr.astype(np.int16)
    return (arr.astype(np.float32) * float(rescale_slope)
            + float(rescale_intercept))


main_series = CtSeries(sample_uid, dicom_files)

hu_volume = np.stack([
    to_hounsfield(main_series.read_pixels(k),
                  main_series.rescale_slope, main_series.rescale_intercept)
    for k in range(main_series.n_slices)
])

n_rows, n_cols = main_series.n_rows, main_series.n_cols
row_spacing, col_spacing = main_series.row_spacing, main_series.col_spacing
slice_spacing_mm = main_series.slice_spacing_mm
iop = np.asarray(main_series.headers[0].ImageOrientationPatient, dtype=float)

print("TAHAP 1 - VOLUME HU")
print("-" * 70)
print(f"Shape volume        : {hu_volume.shape}  (k, baris, kolom)")
print(f"Ukuran image        : {n_rows} x {n_cols}")
print(f"Row spacing (baris) : {row_spacing} mm")
print(f"Col spacing (kolom) : {col_spacing} mm")
print(f"Isotropic in-plane  : {main_series.spacing_is_isotropic}")
print(f"Slice spacing       : {slice_spacing_mm:.4f} mm")
print(f"Rescale             : slope={main_series.rescale_slope} "
      f"intercept={main_series.rescale_intercept}")
print(f"Urutan k            : ascending proyeksi ImagePositionPatient")
print(f"                     k=0 = proyeksi terkecil (bukan 'inferior')")

fov = hu_volume[hu_volume > -2000]
print(f"\nHU min/max (FOV)   : {fov.min():.1f} / {fov.max():.1f}")
print(f"Median HU           : {np.median(fov):.1f}")
if np.median(fov) < -800:
    print("PERINGATAN: median HU sangat rendah. Periksa RescaleSlope/Intercept.")


TAHAP 1 - VOLUME HU
----------------------------------------------------------------------
Shape volume        : (113, 512, 512)  (k, baris, kolom)
Ukuran image        : 512 x 512
Row spacing (baris) : 0.703125 mm
Col spacing (kolom) : 0.703125 mm
Isotropic in-plane  : True
Slice spacing       : 2.5000 mm
Rescale             : slope=1.0 intercept=-1024.0
Urutan k            : ascending proyeksi ImagePositionPatient
                     k=0 = proyeksi terkecil (bukan 'inferior')

HU min/max (FOV)   : -1024.0 / 3071.0
Median HU           : -92.0


## 6. Windowing (TAHAP 2 + 3)

Lung window **[-1350, 150] HU** digunakan sebagai parameter preprocessing yang
telah ditetapkan setelah validasi sample. Window ini global, tidak adaptif,
dan tidak per-cluster.

```
D    = clip(HU, HU_MIN, HU_MAX)
X01  = (D - HU_MIN) / (HU_MAX - HU_MIN)        -> [0, 1]
```

Perbandingan window mediastinal vs lung beserta tabel buktinya dipindahkan ke
`scripts/04_validate_preprocessing.py`.

In [19]:
# --- CELL 6 - TAHAP 2 + 3: WINDOW -----------------------------------------
def apply_window(hu: np.ndarray, hu_min: float, hu_max: float) -> np.ndarray:
    """TAHAP 2 + TAHAP 3: lung window lalu scaling ke [0, 1]."""
    if hu_max <= hu_min:
        raise ValueError(f"hu_max harus > hu_min, diterima {hu_min}..{hu_max}")
    return np.clip((hu - hu_min) / (hu_max - hu_min), 0.0, 1.0).astype(np.float32)


print("TAHAP 2 + 3 - WINDOW")
print("-" * 70)
print(f"HU_MIN / HU_MAX : {WINDOW_HU_MIN} / {WINDOW_HU_MAX}")
print(f"level / width   : {(WINDOW_HU_MIN + WINDOW_HU_MAX) / 2:.0f} / "
      f"{WINDOW_HU_MAX - WINDOW_HU_MIN:.0f}  (untuk metodologi)")
print(f"Adaptive        : tidak dipakai")

windowed_volume = apply_window(hu_volume, WINDOW_HU_MIN, WINDOW_HU_MAX)
print(f"\nVolume windowed : {windowed_volume.shape} {windowed_volume.dtype}")
print(f"Rentang         : {windowed_volume.min():.3f} .. {windowed_volume.max():.3f}")


TAHAP 2 + 3 - WINDOW
----------------------------------------------------------------------
HU_MIN / HU_MAX : -1350.0 / 150.0
level / width   : -600 / 1500  (untuk metodologi)
Adaptive        : tidak dipakai

Volume windowed : (113, 512, 512) float32
Rentang         : 0.000 .. 1.000


## 7. Bounding Box

```
BBox_X_Min / BBox_X_Max  ->  indeks BARIS  (row)
BBox_Y_Min / BBox_Y_Max  ->  indeks KOLOM  (column)
BBox_Z_Min / BBox_Z_Max  ->  indeks SLICE  (k) pada volume terurut
```

Konvensi `BBox_X = baris` dan `BBox_Y = kolom` telah divalidasi sebelumnya
terhadap data LIDC dan digunakan oleh pipeline. Audit pembuktiannya
(`evaluate_convention`, `ring_lung`, `box_solid`) dipindahkan ke
`scripts/04_validate_preprocessing.py` dan tidak dijalankan di notebook ini.

### Margin per-axis

`PixelSpacing` DICOM berisi dua nilai. Margin milimeter dikonversi **terpisah
per sumbu**:

```
margin_row_px = ceil(margin_mm / row_spacing)     -> arah baris
margin_col_px = ceil(margin_mm / col_spacing)     -> arah kolom
```

Rata-rata `(row_spacing + col_spacing) / 2` **tidak** dipakai untuk crop.

In [20]:
# --- CELL 7 - BBOX, MARGIN PER-AXIS, CROP BOX -----------------------------
def map_bbox(x0, x1, y0, y1, convention: str, n_rows: int, n_cols: int):
    """BBox annotation -> (baris_min, baris_max, kolom_min, kolom_max)."""
    if convention == "standard":
        r0, r1, c0, c1 = y0, y1, x0, x1
    elif convention == "transposed":
        r0, r1, c0, c1 = x0, x1, y0, y1
    else:
        raise ValueError(f"konvensi tidak dikenal: {convention!r}")
    return max(0, r0), min(n_rows - 1, r1), max(0, c0), min(n_cols - 1, c1)


def margin_px_from_mm(margin_mm: float, spacing_mm: float, axis: str = "") -> int:
    """Konversi margin mm -> px pada SATU sumbu, dibulatkan ke atas."""
    if spacing_mm <= 0:
        raise ValueError(f"pixel spacing tidak valid: {spacing_mm}")
    return int(np.ceil(margin_mm / spacing_mm))


def margin_px_pair_from_mm(margin_mm: float, row_spacing: float,
                           col_spacing: float) -> tuple[int, int]:
    """Konversi margin mm -> (margin_row_px, margin_col_px), PER SUMBU."""
    if margin_mm < 0:
        raise ValueError(f"margin_mm harus >= 0, diterima {margin_mm}")
    return (margin_px_from_mm(margin_mm, row_spacing, "row"),
            margin_px_from_mm(margin_mm, col_spacing, "col"))


def clip_range(lo: int, hi: int, limit: int) -> tuple[int, int]:
    """Potong rentang indeks ke [0, limit-1]."""
    lo = int(max(0, lo))
    hi = int(min(limit - 1, hi))
    if hi < lo:
        raise ValueError(f"Rentang [{lo}, {hi}] tidak valid terhadap {limit}.")
    return lo, hi


def build_crop_box(row, series: CtSeries, margin_row_px: int = 0,
                   margin_col_px: int = 0) -> dict:
    """Batas crop in-plane dengan margin TERPISAH untuk baris dan kolom.

        baris : r0 - margin_row_px .. r1 + margin_row_px
        kolom : c0 - margin_col_px .. c1 + margin_col_px

    `margin_row_px` dan `margin_col_px` boleh berbeda.
    """
    r0, r1, c0, c1 = map_bbox(
        int(row.BBox_X_Min), int(row.BBox_X_Max),
        int(row.BBox_Y_Min), int(row.BBox_Y_Max),
        BBOX_XY_CONVENTION, series.n_rows, series.n_cols)
    core_r0, core_r1, core_c0, core_c1 = r0, r1, c0, c1

    margin_row_px, margin_col_px = int(margin_row_px), int(margin_col_px)
    if margin_row_px < 0 or margin_col_px < 0:
        raise ValueError(f"margin px harus >= 0, "
                         f"row={margin_row_px}, col={margin_col_px}")

    cr0, cr1 = clip_range(r0 - margin_row_px, r1 + margin_row_px, series.n_rows)
    cc0, cc1 = clip_range(c0 - margin_col_px, c1 + margin_col_px, series.n_cols)

    z0 = int(max(0, int(row.BBox_Z_Min)))
    z1 = int(min(series.n_slices - 1, int(row.BBox_Z_Max)))
    if z1 < z0:
        raise ValueError(f"BBox_Z tidak valid: {z0}..{z1}")
    n_z = z1 - z0 + 1
    k_rel = middle_index(n_z)

    actual_h, actual_w = cr1 - cr0 + 1, cc1 - cc0 + 1
    actual_margin_row_px = core_r0 - cr0
    actual_margin_col_px = core_c0 - cc0

    return {
        "core_r0": core_r0, "core_r1": core_r1,
        "core_c0": core_c0, "core_c1": core_c1,
        "r0": cr0, "r1": cr1, "c0": cc0, "c1": cc1,
        "z0": z0, "z1": z1, "n_slices_z": n_z,
        "k_rel": k_rel, "k_abs": z0 + k_rel,
        "margin_row_px": margin_row_px,
        "margin_col_px": margin_col_px,
        "actual_margin_row_mm": actual_margin_row_px * series.row_spacing,
        "actual_margin_col_mm": actual_margin_col_px * series.col_spacing,
        "clipped_row": bool(actual_margin_row_px != margin_row_px),
        "clipped_col": bool(actual_margin_col_px != margin_col_px),
        "crop_h": actual_h, "crop_w": actual_w,
        "crop_h_mm": actual_h * series.row_spacing,
        "crop_w_mm": actual_w * series.col_spacing,
        "core_h": core_r1 - core_r0 + 1,
        "core_w": core_c1 - core_c0 + 1,
        "rel_core_r0": core_r0 - cr0, "rel_core_r1": core_r1 - cr0,
        "rel_core_c0": core_c0 - cc0, "rel_core_c1": core_c1 - cc0,
    }


print("KONVENSI BBOX")
print("-" * 70)
print(f"konvensi    : {BBOX_XY_CONVENTION}  (BBox_X -> baris, BBox_Y -> kolom)")
print(f"margin      : {CONTEXT_MARGIN_MM:g} mm  ->  {MARGIN_STATUS}")
print(f"  (placeholder; kandidat margin diaudit di scripts/04_validate_preprocessing.py)")


KONVENSI BBOX
----------------------------------------------------------------------
konvensi    : transposed  (BBox_X -> baris, BBox_Y -> kolom)
margin      : 0 mm  ->  BELUM DIKUNCI
  (placeholder; kandidat margin diaudit di scripts/04_validate_preprocessing.py)


## 8. Slice Selection

Aturan terkunci, murni geometris:

```
middle_index = (n_slices - 1) // 2
```

| n | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| index | 0 | 0 | 1 | 1 | 2 | 2 |

Ganjil → slice tengah. Genap → indeks lebih rendah pada urutan slice.

**"Lower-middle" berarti indeks lebih rendah pada urutan hasil sorting
proyeksi `ImagePositionPatient` terhadap normal slice — bukan klaim anatomis
superior/inferior.** Arah normal slice tidak diverifikasi di notebook ini.

In [21]:
# --- CELL 8 - ATURAN PEMILIHAN SLICE --------------------------------------
def middle_index(n_slices: int) -> int:
    """Aturan slice terkunci: (n_slices - 1) // 2.

    Ganjil -> slice tengah. Genap -> indeks lebih rendah pada urutan slice.
    'Indeks lebih rendah' BUKAN berarti 'anatomis inferior'.
    """
    if n_slices < 1:
        raise ValueError(f"n_slices harus >= 1, diterima {n_slices}")
    return (n_slices - 1) // 2


print("ATURAN SLICE (terkunci)")
print("-" * 70)
print(f"{SLICE_SELECTION} : middle_index(n) = (n - 1) // 2")
print(f"{'n':>3} | {'index':>5} | posisi")
print("-" * 34)
for n in range(1, 8):
    k = middle_index(n)
    posisi = "tengah" if n % 2 else f"indeks-lower dari {n}"
    print(f"{n:>3} | {k:>5} | {posisi}")
print("\n'lower-middle' = indeks lebih rendah pada urutan sorting proyeksi IPP,")
print("BUKAN klaim anatomis inferior.")


ATURAN SLICE (terkunci)
----------------------------------------------------------------------
geometric_lower_middle : middle_index(n) = (n - 1) // 2
  n | index | posisi
----------------------------------
  1 |     0 | tengah
  2 |     0 | indeks-lower dari 2
  3 |     1 | tengah
  4 |     1 | indeks-lower dari 4
  5 |     2 | tengah
  6 |     2 | indeks-lower dari 6
  7 |     3 | tengah

'lower-middle' = indeks lebih rendah pada urutan sorting proyeksi IPP,
BUKAN klaim anatomis inferior.


## 9. Crop and Resize

Mengambil slice terpilih dari volume HU, memotong area bbox (plus margin
placeholder), lalu resize ke **224 × 224**.

In [22]:
# --- CELL 9 - RESIZE -------------------------------------------------------
def resize_to(img: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    """Resize 2D (H, W) -> size, interpolasi linear + anti-aliasing.

    Rentang [0, 1] dipertahankan (preserve_range=True).
    """
    return sk_resize(img.astype(np.float32), size, order=1, mode="reflect",
                     anti_aliasing=True, preserve_range=True).astype(np.float32)


print("RESIZE")
print("-" * 70)
print(f"input  : (H, W) windowed, nilai [0, 1]")
print(f"output : {TARGET_SIZE} float32")
print(f"interp : linear (order=1) + anti-aliasing, mode='reflect'")


RESIZE
----------------------------------------------------------------------
input  : (H, W) windowed, nilai [0, 1]
output : (224, 224) float32
interp : linear (order=1) + anti-aliasing, mode='reflect'


## 10. Three Channel — channels_last (HWC)

`DenseNet121` di Keras memakai `data_format="channels_last"`, yaitu input
`(None, 224, 224, 3)`. Karena itu pipeline ini memakai **HWC**:

```
(H, W)  ->  (224, 224)  ->  (224, 224, 3)
grayscale  resize          3 kanal HWC
```

Kanal diisi **replikasi** grayscale (kanal 0 = 1 = 2), bukan kanal semu.

In [23]:
# --- CELL 10 - TAHAP 4: 3 KANAL HWC ---------------------------------------
def to_three_channel(img01: np.ndarray) -> np.ndarray:
    """TAHAP 4 - grayscale (H, W) dalam [0,1] -> (H, W, 3) dalam [0,1].

    Tata letak kanal mengikuti TensorFlow/Keras (channels_last), sehingga
    bentuk keluaran langsung cocok dengan input model tanpa transposisi.
    Kanal diisi replikasi grayscale, bukan kanal semu.
    """
    if img01.ndim != 2:
        raise ValueError(f"diharapkan citra 2D, diterima {img01.shape}")
    return np.repeat(img01[:, :, None], NUM_CHANNELS, axis=2).astype(np.float32)


_probe = to_three_channel(np.zeros((8, 8), np.float32))
print("TAHAP 4 - 3 KANAL")
print("-" * 70)
print(f"input   : (H, W)  grayscale")
print(f"output  : {_probe.shape}  {CHANNEL_LAYOUT} (H, W, C)")
print(f"nilai   : replikasi grayscale, kanal 0 = 1 = 2")
assert _probe.shape == (8, 8, 3), f"shape tak terduga: {_probe.shape}"


TAHAP 4 - 3 KANAL
----------------------------------------------------------------------
input   : (H, W)  grayscale
output  : (8, 8, 3)  channels_last (H, W, C)
nilai   : replikasi grayscale, kanal 0 = 1 = 2


## 11. DenseNet121 Preprocessing

Fungsi **resmi** Keras yang dipanggil langsung, bukan replika manual:

```python
tf.keras.applications.densenet.preprocess_input(input_3ch * 255.0)
```

Fungsi tersebut melakukan `* 255` lalu `/ 255` (saling meniadakan) sebelum
normalisasi mean/std ImageNet, sehingga input `[0,1]` dan `[0,255]` memberi
hasil identik.

`Resize(256) + CenterCrop(224)` ImageNet **sengaja tidak dipakai**: crop
nodule sudah merupakan area input, dan CenterCrop tambahan akan membuang
bagian nodule.

In [24]:
# --- CELL 11 - TAHAP 5: DENSENET121 PREPROCESS_INPUT ----------------------
try:
    import tensorflow as tf
    import keras
    TF_AVAILABLE = True
except ImportError as exc:                                # pragma: no cover
    TF_AVAILABLE = False
    TF_IMPORT_ERROR = exc

if not TF_AVAILABLE:
    raise ImportError(
        "TensorFlow tidak tersedia, jadi preprocessing DenseNet121 tidak "
        "bisa dijalankan.\n\n"
        "Pasang environment sesuai requirements.txt:\n"
        "    python -m venv .venv\n"
        "    .venv\\Scripts\\python -m pip install -r requirements.txt\n\n"
        f"Detail: {TF_IMPORT_ERROR}"
    )

print("TAHAP 5 - DENSENET121 PREPROCESS_INPUT")
print("-" * 70)
print(f"tensorflow : {tf.__version__}")
print(f"keras      : {keras.__version__}")
print(f"fungsi     : tf.keras.applications.densenet.preprocess_input")
print(f"data_format: channels_last")


def keras_densenet_preprocess_input(input_3ch_hwc: np.ndarray) -> np.ndarray:
    """TAHAP 5 - DenseNet121 preprocess_input resmi dari Keras.

    Bentuk input dan output: (H, W, 3) channels_last.
    Fungsi Keras yang dipanggil: x*255 -> x/255 -> (x[c]-mean[c])/std[c]
    (mode="torch" = mean/std ImageNet).
    """
    if input_3ch_hwc.ndim != 3:
        raise ValueError(f"diharapkan (H, W, C), diterima {input_3ch_hwc.shape}")
    if input_3ch_hwc.shape[2] != NUM_CHANNELS:
        raise ValueError(f"diharapkan {NUM_CHANNELS} kanal, "
                         f"diterima {input_3ch_hwc.shape}")
    x255 = (input_3ch_hwc * 255.0).astype(np.float32)
    return np.asarray(
        tf.keras.applications.densenet.preprocess_input(
            x255, data_format="channels_last"),
        dtype=np.float32)


def run_pipeline(row, series: CtSeries, margin_row_px: int = 0,
                 margin_col_px: int = 0,
                 volume_hu: np.ndarray | None = None,
                 size=TARGET_SIZE) -> dict:
    """Jalankan seluruh tahap untuk satu cluster.

    HU -> lung window -> [0,1] -> resize -> 3 kanal (H,W,3)
       -> densenet.preprocess_input -> (224, 224, 3)
    """
    box = build_crop_box(row, series, margin_row_px, margin_col_px)
    k = box["k_abs"]
    hu_slice = (volume_hu[k] if volume_hu is not None
                else to_hounsfield(series.read_pixels(k),
                                   series.rescale_slope,
                                   series.rescale_intercept))
    sl = hu_slice[box["r0"]:box["r1"] + 1, box["c0"]:box["c1"] + 1]

    windowed = apply_window(sl, WINDOW_HU_MIN, WINDOW_HU_MAX)   # tahap 2+3
    resized = resize_to(windowed, size)
    input_3ch = to_three_channel(resized)                       # tahap 4
    model_input = keras_densenet_preprocess_input(input_3ch)    # tahap 5

    # bentuk harus channels_last
    assert input_3ch.shape == (size[0], size[1], NUM_CHANNELS), \
        f"input_3ch harus {(size[0], size[1], NUM_CHANNELS)}, " \
        f"didapat {input_3ch.shape}"
    assert model_input.shape == (size[0], size[1], NUM_CHANNELS), \
        f"model_input harus {(size[0], size[1], NUM_CHANNELS)}, " \
        f"didapat {model_input.shape}"

    chans = [input_3ch[..., c] for c in range(NUM_CHANNELS)]
    return {
        "box": box, "hu_slice": sl, "windowed": windowed,
        "resized": resized, "input_3ch": input_3ch,
        "model_input": model_input,
        "input_3ch_shape": tuple(input_3ch.shape),
        "model_input_shape": tuple(model_input.shape),
        "upsample_factor": max(size) / max(box["crop_h"], box["crop_w"]),
        "kanal_identik": bool(np.array_equal(chans[0], chans[1])
                              and np.array_equal(chans[1], chans[2])),
    }


TAHAP 5 - DENSENET121 PREPROCESS_INPUT
----------------------------------------------------------------------
tensorflow : 2.21.0
keras      : 3.15.1
fungsi     : tf.keras.applications.densenet.preprocess_input
data_format: channels_last


## 12. Sample Validation

Menjalankan pipeline pada beberapa cluster representatif dari seri sample,
lalu memvalidasi bentuk, rentang nilai, dan integritas numerik.

In [25]:
# --- CELL 12.1 - JALANKAN PIPELINE PADA SAMPLE ---------------------------
# PLACEHOLDER - BUKAN KEPUTUSAN METODOLOGI
MARGIN_ROW_PX, MARGIN_COL_PX = margin_px_pair_from_mm(
    CONTEXT_MARGIN_MM, main_series.row_spacing, main_series.col_spacing)
print(f"Margin placeholder : {CONTEXT_MARGIN_MM:g} mm -> "
      f"{MARGIN_ROW_PX} px baris, {MARGIN_COL_PX} px kolom  [{MARGIN_STATUS}]\n")

results_by_id = {}
summary_rows = []

for _, row in sample_ann.iterrows():
    out = run_pipeline(row, main_series, MARGIN_ROW_PX, MARGIN_COL_PX,
                       volume_hu=hu_volume)
    results_by_id[row.NoduleID] = out
    b = out["box"]
    fov_vals = out["hu_slice"][out["hu_slice"] > -2000]

    summary_rows.append({
        "NoduleID": row.NoduleID,
        "ClusterIndex": int(row.ClusterIndex),
        "Label": row.Label,
        "n_slice_crop": b["n_slices_z"],
        "k_abs": b["k_abs"],
        "crop_px": f"{b['crop_h']}x{b['crop_w']}",
        "crop_mm": f"{b['crop_h_mm']:.0f}x{b['crop_w_mm']:.0f}",
        "upsample": round(out["upsample_factor"], 1),
        "HU_min": float(fov_vals.min()) if fov_vals.size else float("nan"),
        "HU_max": float(fov_vals.max()) if fov_vals.size else float("nan"),
        "shape_3ch": str(out["input_3ch"].shape),
        "shape_model": str(out["model_input"].shape),
        "range_3ch": f"[{out['input_3ch'].min():.3f}, {out['input_3ch'].max():.3f}]",
        "range_model": f"[{out['model_input'].min():.2f}, {out['model_input'].max():.2f}]",
    })

summary = pd.DataFrame(summary_rows)
print("RINGKASAN PREPROCESSING (seri sample)")
print("-" * 100)
display(summary)


Margin placeholder : 0 mm -> 0 px baris, 0 px kolom  [BELUM DIKUNCI]

RINGKASAN PREPROCESSING (seri sample)
----------------------------------------------------------------------------------------------------


,NoduleID,ClusterIndex,Label,n_slice_crop,k_abs,crop_px,crop_mm,upsample,HU_min,HU_max,shape_3ch,shape_model,range_3ch,range_model
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016...,1,Benign,2,51,8x8,6x6,28.0,-629.0,-26.0,"(224, 224, 3)","(224, 224, 3)","[0.482, 0.883]","[-0.01, 2.12]"
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016...,2,Benign,2,52,7x8,5x6,28.0,-638.0,-267.0,"(224, 224, 3)","(224, 224, 3)","[0.475, 0.722]","[-0.04, 1.40]"
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016...,3,Benign,1,57,8x8,6x6,28.0,-715.0,-46.0,"(224, 224, 3)","(224, 224, 3)","[0.425, 0.869]","[-0.26, 2.06]"
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016...,4,Malignant,11,59,32x34,22x24,6.6,-800.0,106.0,"(224, 224, 3)","(224, 224, 3)","[0.368, 0.970]","[-0.51, 2.51]"
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016...,5,Benign,3,60,12x11,8x8,18.7,-899.0,6.0,"(224, 224, 3)","(224, 224, 3)","[0.301, 0.902]","[-0.80, 2.21]"
5,1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016...,6,Exclude,3,67,16x15,11x11,14.0,-789.0,48.0,"(224, 224, 3)","(224, 224, 3)","[0.374, 0.931]","[-0.48, 2.33]"
6,1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016...,7,Benign,3,70,13x13,9x9,17.2,-780.0,-17.0,"(224, 224, 3)","(224, 224, 3)","[0.380, 0.888]","[-0.46, 2.14]"
7,1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016...,8,Malignant,3,75,15x13,11x9,14.9,-792.0,-10.0,"(224, 224, 3)","(224, 224, 3)","[0.372, 0.893]","[-0.49, 2.16]"
8,1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016...,9,Benign,2,90,8x8,6x6,28.0,-805.0,-63.0,"(224, 224, 3)","(224, 224, 3)","[0.364, 0.858]","[-0.53, 2.01]"
9,1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016...,10,Benign,2,91,9x10,6x7,22.4,-752.0,-266.0,"(224, 224, 3)","(224, 224, 3)","[0.399, 0.721]","[-0.38, 1.40]"


In [26]:
# --- CELL 12.2 - VALIDASI KESELURUHAN ------------------------------------
fails = []


def check(name, ok, detail=""):
    print(f"  [{'OK ' if ok else 'GAGAL'}] {name:<44} {detail}")
    if not ok:
        fails.append(name)


n = len(results_by_id)
check("jumlah cluster diproses", n > 0, f"{n} cluster")

for nid, o in results_by_id.items():
    b = o["box"]
    check(f"{nid}: shape 3 kanal HWC",
          o["input_3ch"].shape == (224, 224, 3), str(o["input_3ch"].shape))
    check(f"{nid}: shape model input HWC",
          o["model_input"].shape == (224, 224, 3), str(o["model_input"].shape))
    check(f"{nid}: dtype float32",
          o["input_3ch"].dtype == np.float32
          and o["model_input"].dtype == np.float32,
          f"{o['input_3ch'].dtype} / {o['model_input'].dtype}")
    check(f"{nid}: tanpa NaN/Inf",
          bool(np.isfinite(o["model_input"]).all()),
          f"min {o['model_input'].min():.3f} max {o['model_input'].max():.3f}")
    check(f"{nid}: range 3 kanal dalam [0,1]",
          0.0 <= o["input_3ch"].min() and o["input_3ch"].max() <= 1.0,
          f"[{o['input_3ch'].min():.3f}, {o['input_3ch'].max():.3f}]")
    check(f"{nid}: kanal identik (replikasi)", o["kanal_identik"], "ch0=ch1=ch2")
    check(f"{nid}: k = (n-1)//2",
          b["k_abs"] == b["z0"] + middle_index(b["n_slices_z"]),
          f"k={b['k_abs']} dari {b['n_slices_z']} slice")
    if CONTEXT_MARGIN_MM > 0:
        check(f"{nid}: margin baris >= diminta",
              b["actual_margin_row_mm"] >= CONTEXT_MARGIN_MM - 1e-6,
              f"{b['actual_margin_row_mm']:.2f} mm")
        check(f"{nid}: margin kolom >= diminta",
              b["actual_margin_col_mm"] >= CONTEXT_MARGIN_MM - 1e-6,
              f"{b['actual_margin_col_mm']:.2f} mm")

if fails:
    raise AssertionError(f"Validasi sample gagal: {fails}")
print(f"\nSemua {n} cluster lolos validasi.")


  [OK ] jumlah cluster diproses                      10 cluster
  [OK ] 1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016904899083676284739_N001: shape 3 kanal HWC (224, 224, 3)
  [OK ] 1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016904899083676284739_N001: shape model input HWC (224, 224, 3)
  [OK ] 1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016904899083676284739_N001: dtype float32 float32 / float32
  [OK ] 1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016904899083676284739_N001: tanpa NaN/Inf min -0.015 max 2.118
  [OK ] 1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016904899083676284739_N001: range 3 kanal dalam [0,1] [0.482, 0.883]
  [OK ] 1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016904899083676284739_N001: kanal identik (replikasi) ch0=ch1=ch2
  [OK ] 1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016904899083676284739_N001: k = (n-1)//2 k=51 dari 2 slice
  [OK ] 1.3.6.1.4.1.14519.5.2.1.6279.6001.616033753016904899083676284739_N002: shape 3 kanal HWC (224, 224, 3)
  [OK ] 1.3.6.1.4.1.14519.5

In [27]:
# --- CELL 12.3 - VISUALISASI SAMPLE ---------------------------------------
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def fov_range(arr):
    v = arr[arr > -2000]
    return (float(v.min()), float(v.max())) if v.size else (float("nan"),) * 2


def plot_sample(row, out, save=True):
    """Enam panel: slice asli, HU, HU+bbox, crop, resize, input model."""
    b = out["box"]
    k = b["k_abs"]
    hu_min, hu_max = fov_range(out["hu_slice"])

    fig, axes = plt.subplots(1, 6, figsize=(25, 4.8))
    raw = main_series.read_pixels(k)

    axes[0].imshow(raw, cmap="gray")
    axes[0].set_title("1. pixel_array (DICOM)", fontsize=9)

    axes[1].imshow(hu_volume[k], cmap="gray", vmin=-1000, vmax=-400)
    axes[1].set_title(f"2. HU (rescale)\n{fov_range(hu_volume[k])[0]:.0f} .. "
                      f"{fov_range(hu_volume[k])[1]:.0f} HU", fontsize=9)

    axes[2].imshow(hu_volume[k], cmap="gray", vmin=-1000, vmax=-400)
    axes[2].add_patch(Rectangle((b["c0"], b["r0"]), b["c1"] - b["c0"],
                                b["r1"] - b["r0"], fill=False, ec="red", lw=1.5))
    axes[2].scatter([(b["core_c0"] + b["core_c1"]) / 2],
                    [(b["core_r0"] + b["core_r1"]) / 2],
                    marker="+", s=120, color="yellow", linewidths=1.3)
    axes[2].set_title("3. HU + bounding box", fontsize=9)

    axes[3].imshow(out["windowed"], cmap="gray", vmin=0, vmax=1)
    axes[3].set_title(f"4. Crop window\n{b['crop_h']}x{b['crop_w']} px "
                      f"({b['crop_h_mm']:.0f}x{b['crop_w_mm']:.0f} mm)", fontsize=9)

    axes[4].imshow(out["resized"], cmap="gray", vmin=0, vmax=1)
    axes[4].set_title(f"5. Resize {TARGET_SIZE[0]}x{TARGET_SIZE[1]}\n"
                      f"upsample x{out['upsample_factor']:.1f}", fontsize=9)

    # HWC: kanal pada sumbu terakhir -> rata-rata(axis=2)
    axes[5].imshow(out["model_input"].mean(axis=2), cmap="gray",
                   vmin=-2.5, vmax=2.7)
    axes[5].set_title(f"6. Input model {out['model_input_shape']}\n"
                      f"densenet.preprocess_input (mean kanal)", fontsize=9)

    for ax in axes:
        ax.axis("off")
    fig.suptitle(
        f"{row.NoduleID}  |  PatientID={row.PatientID}  |  "
        f"Label={row.Label}  (MeanMalignancy={row.MeanMalignancy})\n"
        f"SeriesInstanceUID={row.SeriesInstanceUID}  |  "
        f"n slice crop={b['n_slices_z']}, k=({b['n_slices_z']}-1)//2="
        f"{b['k_rel']} -> k={k}  |  HU crop {hu_min:.0f}..{hu_max:.0f}  |  "
        f"margin {b['margin_row_px']} px baris / {b['margin_col_px']} px kolom "
        f"({MARGIN_STATUS})",
        fontsize=9)
    fig.tight_layout(rect=(0, 0, 1, 0.90))

    path = None
    if save:
        safe = row.NoduleID.replace(".", "_")[-60:]
        path = FIGURE_DIR / f"sample_cluster{int(row.ClusterIndex):02d}_{safe}.png"
        fig.savefig(path, dpi=110, bbox_inches="tight")
    plt.close(fig)
    return path


# pilih sample berdasarkan ukuran bbox (bukan label)
ordered = sample_ann.assign(
    _core_px=(sample_ann["BBox_X_Max"] - sample_ann["BBox_X_Min"] + 1)
    + (sample_ann["BBox_Y_Max"] - sample_ann["BBox_Y_Min"] + 1)
).sort_values("_core_px")

if len(ordered) > 1:
    idx = np.linspace(0, len(ordered) - 1, N_SAMPLE_FIGURES).round().astype(int)
    chosen = ordered.iloc[np.unique(idx)]
else:
    chosen = ordered

figure_paths = []
for _, row in chosen.iterrows():
    p = plot_sample(row, results_by_id[row.NoduleID])
    figure_paths.append(p)
    print(f"  {p.name}")

print(f"\n{len(figure_paths)} figure sample disimpan ke {FIGURE_RELPATH}")


  sample_cluster02__1_14519_5_2_1_6279_6001_616033753016904899083676284739_N002.png
  sample_cluster04__1_14519_5_2_1_6279_6001_616033753016904899083676284739_N004.png

2 figure sample disimpan ke results\preprocessing_sample


## 13. DenseNet121 Verification

Pemeriksaan singkat: bentuk pipeline, bentuk model, kesetaraan dengan fungsi
Keras pada input yang sama, dan forward pass.

In [28]:
# --- CELL 13 - VERIFIKASI DENSENET121 -------------------------------------
model = tf.keras.applications.DenseNet121(weights="imagenet")
print("VERIFIKASI DENSENET121")
print("-" * 70)
print(f"model input_shape  : {model.input_shape}")
print(f"model output_shape : {model.output_shape}")
print(f"parameter          : {model.count_params():,}")

# bentuk pipeline
for nid, o in results_by_id.items():
    assert o["input_3ch"].shape == (224, 224, 3)
    assert o["model_input"].shape == (224, 224, 3)
print(f"pipeline 3 kanal   : {INPUT_SHAPE_HWC}  (channels_last)")
print(f"pipeline model     : {INPUT_SHAPE_HWC}")

# bentuk model harus cocok
if tuple(model.input_shape) != INPUT_SHAPE_NHWC:
    raise ValueError(f"Bentuk input model {model.input_shape} tidak cocok "
                     f"dengan {INPUT_SHAPE_NHWC}.")
print(f"model cocok        : {INPUT_SHAPE_NHWC}  OK")

# kesetaraan dengan fungsi Keras pada INPUT YANG SAMA
rng = np.random.default_rng(11)
probe = rng.random(INPUT_SHAPE_HWC, dtype=np.float32)
mine = keras_densenet_preprocess_input(probe)
ref = np.asarray(
    tf.keras.applications.densenet.preprocess_input(
        (probe * 255.0).copy(), data_format="channels_last"),
    dtype=np.float32)
max_abs_diff = float(np.abs(mine - ref).max())
print(f"\npreprocess_input equivalence: max_abs_diff = {max_abs_diff:.3e}")
if max_abs_diff > KERAS_TOLERANCE:
    raise AssertionError(f"Preprocessing tidak cocok dengan fungsi Keras "
                         f"(max_abs_diff={max_abs_diff:.3e} > {KERAS_TOLERANCE})")
print(f"toleransi terpenuhi (<= {KERAS_TOLERANCE:g})")

# forward pass pada output pipeline
logits = model.predict(results_by_id[sample_ann['NoduleID'].iloc[0]]["model_input"][None, ...],
                       verbose=0)
print(f"\nforward pass      : logits {logits.shape}, "
      f"rentang {logits.min():.2f} .. {logits.max():.2f}")
if logits.shape != (1, 1000):
    raise AssertionError(f"Forward pass menghasilkan {logits.shape}")
print("\nSemua verifikasi lolos.")


VERIFIKASI DENSENET121
----------------------------------------------------------------------
model input_shape  : (None, 224, 224, 3)
model output_shape : (None, 1000)
parameter          : 8,062,504
pipeline 3 kanal   : (224, 224, 3)  (channels_last)
pipeline model     : (224, 224, 3)
model cocok        : (None, 224, 224, 3)  OK

preprocess_input equivalence: max_abs_diff = 0.000e+00
toleransi terpenuhi (<= 1e-06)


2026-10-01 12:28:46,626 - TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.



forward pass      : logits (1, 1000), rentang 0.00 .. 0.16

Semua verifikasi lolos.


## 14. Final Configuration Summary

Parameter yang benar-benar dipakai pada run ini.

In [29]:
# --- CELL 14 - RINGKASAN PARAMETER ---------------------------------------
b_first = next(iter(results_by_id.values()))["box"]

params = {
    "Window (TERKUNCI)": f"lung HU_MIN={WINDOW_HU_MIN} HU_MAX={WINDOW_HU_MAX}",
    "  level / width setara": f"{(WINDOW_HU_MIN + WINDOW_HU_MAX) / 2:.0f} / "
                             f"{WINDOW_HU_MAX - WINDOW_HU_MIN:.0f}",
    "  adaptive": "tidak dipakai",
    "Konvensi bbox (TERKUNCI)": f"{BBOX_XY_CONVENTION} (BBox_X=baris, BBox_Y=kolom)",
    "Aturan slice (TERKUNCI)": f"middle_index = (n-1)//2  ({SLICE_SELECTION})",
    "Framework (TERKUNCI)": "TensorFlow / Keras",
    "Model (TERKUNCI)": "DenseNet121(weights='imagenet')",
    "Ukuran input (TERKUNCI)": f"{INPUT_SHAPE_HWC}  channels_last (HWC)",
    "  batch input model": f"{INPUT_SHAPE_NHWC}",
    "Preprocessing (TERKUNCI)": "tf.keras.applications.densenet.preprocess_input",
    "  max_abs_diff vs Keras": f"{max_abs_diff:.3e}  (toleransi {KERAS_TOLERANCE:g})",
    "  ImageNet Resize/CenterCrop": "TIDAK dipakai (disengaja)",
    "Channel": f"replikasi grayscale ke {NUM_CHANNELS} kanal, sumbu terakhir",
    "---": "---",
    "Margin konteks": f"{MARGIN_STATUS}",
    "  nilai dipakai": f"{CONTEXT_MARGIN_MM:g} mm (PLACEHOLDER, bukan keputusan)",
    "  konversi": f"per-axis: {MARGIN_ROW_PX} px baris, {MARGIN_COL_PX} px kolom",
    "  audit kandidat": "scripts/04_validate_preprocessing.py",
    "---": "---",
    "Seri sample": sample_uid,
    "PatientID": sample_ann["PatientID"].iloc[0],
    "Cluster diproses": len(results_by_id),
    "Ukuran image": f"{n_rows} x {n_cols}",
    "Row spacing (mm)": row_spacing,
    "Col spacing (mm)": col_spacing,
    "Isotropic in-plane": main_series.spacing_is_isotropic,
    "Slice spacing (mm)": round(slice_spacing_mm, 4),
    "Slice thickness (mm)": main_series.slice_thickness,
    "Jumlah slice volume": main_series.n_slices,
    "Rescale slope/intercept": f"{main_series.rescale_slope} / "
                               f"{main_series.rescale_intercept}",
    "ImageOrientationPatient": [float(v) for v in iop],
    "Urutan k": "ascending proyeksi ImagePositionPatient onto normal "
                "(k=0 = proyeksi terkecil, bukan 'inferior')",
    "---": "---",
    "TensorFlow": tf.__version__,
    "Keras": keras.__version__,
    "NumPy": np.__version__,
    "pandas": pd.__version__,
    "pydicom": pydicom.__version__,
    "Python": sys.version.split()[0],
}

width = max(len(k) for k in params)
print("PARAMETER AKHIR")
print("=" * (width + 46))
for key, value in params.items():
    print(f"{key:<{width}} : {value}")

# assertion: margin tetap belum dikunci
assert CONTEXT_MARGIN_MM == 0.0, "Margin harus tetap placeholder 0 mm."
print(f"\nMargin konteks: {MARGIN_STATUS} — nilai final belum dipilih.")


PARAMETER AKHIR
Window (TERKUNCI)            : lung HU_MIN=-1350.0 HU_MAX=150.0
  level / width setara       : -600 / 1500
  adaptive                   : tidak dipakai
Konvensi bbox (TERKUNCI)     : transposed (BBox_X=baris, BBox_Y=kolom)
Aturan slice (TERKUNCI)      : middle_index = (n-1)//2  (geometric_lower_middle)
Framework (TERKUNCI)         : TensorFlow / Keras
Model (TERKUNCI)             : DenseNet121(weights='imagenet')
Ukuran input (TERKUNCI)      : (224, 224, 3)  channels_last (HWC)
  batch input model          : (None, 224, 224, 3)
Preprocessing (TERKUNCI)     : tf.keras.applications.densenet.preprocess_input
  max_abs_diff vs Keras      : 0.000e+00  (toleransi 1e-06)
  ImageNet Resize/CenterCrop : TIDAK dipakai (disengaja)
Channel                      : replikasi grayscale ke 3 kanal, sumbu terakhir
---                          : ---
Margin konteks               : BELUM DIKUNCI
  nilai dipakai              : 0 mm (PLACEHOLDER, bukan keputusan)
  konversi                   